# **PySpark + Iceberg Lakehouse Lab**
Connect to the Iceberg REST catalog and store table data in SeaweedFS S3.

In [2]:
# Import the SparkSession entry point to programmatically configure and initialize Spark
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    # Name the application for identification in the Spark UI and cluster resource manager
    .appName("LakehouseLab")
    # 1. Enable Apache Iceberg SQL extensions (adds custom syntax like MERGE INTO, CREATE OR REPLACE, time travel, etc.)
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
    )
    # 2. Register a new Iceberg catalog named 'lake' using Spark's SparkCatalog implementation
    .config("spark.sql.catalog.lake", "org.apache.iceberg.spark.SparkCatalog")
    # 3. Set the catalog backend type to 'rest' (communicates with an Iceberg REST catalog server for table metadata)
    .config("spark.sql.catalog.lake.type", "rest")
    # 4. Specify the URI endpoint of the Iceberg REST catalog service container
    .config("spark.sql.catalog.lake.uri", "http://iceberg-rest:8181")
    # 5. Define the root S3 warehouse directory path where Iceberg data files and manifests will be stored
    .config("spark.sql.catalog.lake.warehouse", "s3://warehouse/")
    # 6. Configure the IO implementation class for reading/writing files via S3 APIs
    .config(
        "spark.sql.catalog.lake.io-impl",
        "org.apache.iceberg.aws.s3.S3FileIO",
    )
    # 7. Point to the S3-compatible object storage endpoint (e.g., SeaweedFS or MinIO)
    .config("spark.sql.catalog.lake.s3.endpoint", "http://seaweedfs:8333")
    # 8. Force path-style access for S3 requests (required for many local/custom S3-compatible stores instead of virtual-hosted style)
    .config("spark.sql.catalog.lake.s3.path-style-access", "true")
    # 9. Provide the S3 storage authentication credentials (Access Key ID)
    .config("spark.sql.catalog.lake.s3.access-key-id", "lakehouse")
    # 10. Provide the S3 storage authentication credentials (Secret Access Key)
    .config("spark.sql.catalog.lake.s3.secret-access-key", "lakehouse-secret")
    # --- CRITICAL: Iceberg catalog-level S3 region settings ---
    .config("spark.sql.catalog.lake.s3.region", "us-east-1")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.endpoint", "http://seaweedfs:8333")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    # -----------------------------------------------------------
    # 11. Set 'lake' as the default catalog so tables can be referenced directly without prefixing the catalog name
    .config("spark.sql.defaultCatalog", "lake")
    # Materialize or retrieve the configured SparkSession instance
    .getOrCreate()
)

# Reduce noisy Spark execution logs by setting the root logger level to WARN instead of INFO
spark.sparkContext.setLogLevel("WARN")

# Print the active PySpark runtime version to verify successful initialization
print("Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/16 12:26:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 3.5.6


## **Register a Spark SQL cell magic**
`ipython-sql` is not a SparkSession adapter. This small magic forwards each `%%sql` cell to `spark.sql()`.

In [3]:
from IPython.core.magic import register_cell_magic

@register_cell_magic
def sql(line, cell):
    result = spark.sql(cell)
    return result.show(100, truncate=False)

## **Creating a Database and an Iceberg Table**

In [4]:
%%sql
CREATE DATABASE IF NOT EXISTS accounts;

++
||
++
++



In [5]:
spark.catalog.listDatabases()

[Database(name='accounts', catalog='lake', description=None, locationUri='s3://warehouse/accounts')]

In [6]:
%%sql
CREATE TABLE IF NOT EXISTS accounts.users (
    id INT,
    first_name VARCHAR(25),
    last_name VARCHAR(25),
    email VARCHAR(50)
)
USING iceberg
PARTITIONED BY (truncate(1, last_name))

++
||
++
++



1. **`CREATE TABLE IF NOT EXISTS accounts.users`**:
* Creates a new table named `users` inside the `accounts` database/namespace (provided it doesn't already exist).


2. **Column Definitions**:
* Defines standard relational schema fields: `id` (integer), `first_name` (variable-length string up to 25 chars), `last_name` (string up to 25 chars), and `email` (string up to 50 chars).


3. **`USING iceberg`**:
* Instructs Spark to create this table using the **Apache Iceberg** table format instead of the default Hive or Parquet formats. This enables ACID transactions, hidden partitioning, and time-travel capabilities on top of your underlying object storage.


4. **`PARTITIONED BY (truncate(1, last_name))`**:
* Leverages Iceberg's **hidden partitioning** feature with a **truncate transform**.
* `truncate(1, last_name)` tells Iceberg to partition the table based on the **first letter** of the `last_name` column (length 1).
* **Why use partition transforms?** In traditional hive-style partitioning, you would have to add a physical derived column to your dataset. With Iceberg, users query the table using regular filters (e.g., `WHERE last_name = 'Smith'`), and Iceberg automatically applies partition pruning under the hood based on the truncated transformation.

In [7]:
spark.catalog.listTables(dbName='accounts')

[Table(name='users', catalog='lake', namespace=['accounts'], description=None, tableType='MANAGED', isTemporary=False)]

In [8]:
%%sql
SHOW TABLES IN accounts;

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|accounts |users    |false      |
+---------+---------+-----------+



In [9]:
%%sql
DROP TABLE IF EXISTS accounts.users;

++
||
++
++



In [10]:
spark.catalog.listTables(dbName='accounts')

[]

In [11]:
%%sql
SHOW TABLES IN accounts;

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+



In [12]:
%%sql
CREATE TABLE IF NOT EXISTS accounts.users (
    id INT,
    first_name VARCHAR(25),
    last_name VARCHAR(25),
    email VARCHAR(50)
)
USING iceberg
PARTITIONED BY (truncate(1, last_name));

++
||
++
++



## **Adding Records to the Table**

In [13]:
%%sql
INSERT INTO accounts.users (id, first_name, last_name, email)
VALUES (1, 'Juli', 'Arthars', 'jarthars0@mit.edu'),
(2, 'Matthiew', 'Hurley', 'mhurley1@narod.ru'),
(3, 'Lena', 'Westcarr', 'lwestcarr2@jimdo.com'),
(4, 'Reagen', 'Josifovitz', 'rjosifovitz3@hexun.com'),
(5, 'Ogden', 'Janecek', 'ojanecek4@yellowbook.com'),
(6, 'Calypso', 'McMurrugh', 'cmcmurrugh5@cam.ac.uk'),
(7, 'Dru', 'Garces', 'dgarces6@privacy.gov.au'),
(8, 'Jack', 'Matschoss', 'jmatschoss7@latimes.com'),
(9, 'Patty', 'Furnell', 'pfurnell8@yelp.com'),
(10, 'Lyndsie', 'Speeks', 'lspeeks9@skyrock.com');

++
||
++
++



## **Querying the Table with PyIceberg**

In [14]:
%%sql
SELECT *
FROM accounts.users;

+---+----------+----------+------------------------+
|id |first_name|last_name |email                   |
+---+----------+----------+------------------------+
|4  |Reagen    |Josifovitz|rjosifovitz3@hexun.com  |
|5  |Ogden     |Janecek   |ojanecek4@yellowbook.com|
|3  |Lena      |Westcarr  |lwestcarr2@jimdo.com    |
|7  |Dru       |Garces    |dgarces6@privacy.gov.au |
|2  |Matthiew  |Hurley    |mhurley1@narod.ru       |
|6  |Calypso   |McMurrugh |cmcmurrugh5@cam.ac.uk   |
|8  |Jack      |Matschoss |jmatschoss7@latimes.com |
|1  |Juli      |Arthars   |jarthars0@mit.edu       |
|9  |Patty     |Furnell   |pfurnell8@yelp.com      |
|10 |Lyndsie   |Speeks    |lspeeks9@skyrock.com    |
+---+----------+----------+------------------------+



In [15]:
from pyiceberg.catalog import load_catalog

In [18]:
# Load the REST catalog
catalog = load_catalog(
    'lake',
    **{
        "type": "rest",
        "uri": "http://iceberg-rest:8181",
        "warehouse": "s3://warehouse/",
        "io-impl": "org.apache.iceberg.aws.s3.S3FileIO",
        "s3.endpoint": "http://seaweedfs:8333",
        "s3.path-style-access": "true",
        "s3.access-key-id": "lakehouse",
        "s3.secret-access-key": "lakehouse-secret",
        "s3.region": "us-east-1",
    },
)

# Load the users table
tb1 = catalog.load_table('accounts.users')

# Create a scan with filter on last_name
sc = tb1.scan(row_filter="last_name LIKE 'M%' ")

# Convert the scan results into a Pandas dataframe
df = sc.to_arrow().to_pandas()

df

,id,first_name,last_name,email
0,6,Calypso,McMurrugh,cmcmurrugh5@cam.ac.uk
1,8,Jack,Matschoss,jmatschoss7@latimes.com


## **Working with PyIceberg CLI**

- **docker exec -it lakehouse-jupyter pyiceberg --catalog lake list ==> accounts**
-  **docker exec -it lakehouse-jupyter pyiceberg --catalog lake list accounts ==> accounts.users**
-   **docker exec -it lakehouse-jupyter pyiceberg --catalog lake describe --entity=table accounts.users =>**
```
Table format version  2                                                                                                                                                   
Metadata location     s3://warehouse/accounts/users/metadata/00001-93f4412b-ab98-43aa-827a-71752351bf9f.metadata.json                                                     
Table UUID            63801c3d-029a-4531-b752-237d0b0190e0                                                                                                                
Last Updated          1794832031250                                                                                                                                       
Partition spec        [                                                                                                                                                   
                        1000: last_name_trunc: truncate[1](3)                                                                                                             
                      ]                                                                                                                                                   
Sort order            []                                                                                                                                                  
Current schema        Schema, id=0                                                                                                                                        
                      ├── 1: id: optional int                                                                                                                             
                      ├── 2: first_name: optional string                                                                                                                  
                      ├── 3: last_name: optional string                                                                                                                   
                      └── 4: email: optional string                                                                                                                       
Current snapshot      Operation.APPEND: id=7764489527232125832, schema_id=0                                                                                               
Snapshots             Snapshots                                                                                                                                           
                      └── Snapshot 7764489527232125832, schema 0:                                                                                                         
                          s3://warehouse/accounts/users/metadata/snap-7764489527232125832-1-0c87745a-be86-44c0-a55f-9872630d940c.avro                                     
Properties            owner                            root                                                                                                               
                      write.parquet.compression-codec  zstd                                                                                                            
```